# Job/Internship Prep Coach Agent
## GDG AI Study Jam

What we're building: an AI agent that takes a job description and your resume, then gives you mock interview questions, skill gap analysis, and a study plan.

What you'll learn:
- How AI agents differ from chatbots
- How to define tools an agent can use
- How to build and run an agent with Google ADK

## Section 1: Setup

In [ ]:
# Cell 1.1 - Install dependencies
!pip install google-adk google-generativeai -q

In [ ]:
# Cell 1.2 - API Key setup
import os

# We'll use the instructor's key for the live demo
# You can get your own free key at: aistudio.google.com
os.environ["GOOGLE_API_KEY"] = "YOUR_KEY_HERE"  # students fill this in

In [ ]:
# Cell 1.3 - Imports
from google.adk.agents import Agent
from google.adk.tools import FunctionTool
from google.adk.runners import InMemoryRunner
from google.genai import types

## What is an Agent?

A chatbot responds to one message at a time.

An agent can:
  - Use tools (functions it can call)
  - Reason across multiple steps
  - Decide WHICH tool to use and WHEN

Our agent will:
  1. Analyze the job description → extract required skills
  2. Analyze the resume → extract candidate skills
  3. Compare them → find skill gaps
  4. Generate interview questions based on the role
  5. Build a prep plan

**The key: we define the tools, the agent decides how to use them.**

## Section 3: Define the Tools

In [ ]:
# Cell 3.1 - Tool: Analyze Job Description
def analyze_job_description(job_description: str) -> dict:
    """
    Extracts key requirements from a job description.
    Returns required skills, role title, and company level.
    
    In a real app this could call another AI or a parser.
    For now, we return structured data for the agent to work with.
    """
    return {
        "raw_description": job_description,
        "instruction": "Extract: role title, required technical skills, soft skills, and experience level."
    }

In [ ]:
# Cell 3.2 - Tool: Analyze Resume
def analyze_resume(resume_text: str) -> dict:
    """
    Extracts candidate's skills and experience from resume text.
    """
    return {
        "raw_resume": resume_text,
        "instruction": "Extract: candidate's technical skills, projects, experience, and education."
    }

In [ ]:
# Cell 3.3 - Tool: Generate Interview Questions
def generate_interview_questions(role: str, focus_areas: str) -> dict:
    """
    Generates mock interview questions for a given role and focus areas.
    """
    return {
        "role": role,
        "focus_areas": focus_areas,
        "instruction": "Generate 5 technical and 3 behavioral interview questions."
    }

In [ ]:
# Cell 3.4 - Tool: Create Prep Plan
def create_prep_plan(skill_gaps: str, weeks_until_deadline: int = 4) -> dict:
    """
    Creates a week-by-week study plan to address skill gaps.
    """
    return {
        "skill_gaps": skill_gaps,
        "weeks": weeks_until_deadline,
        "instruction": f"Create a {weeks_until_deadline}-week prep plan to address these gaps."
    }

## Section 4: Register Tools with ADK

In [ ]:
# Cell 4.1 - Wrap functions as ADK tools
tools = [
    FunctionTool(analyze_job_description),
    FunctionTool(analyze_resume),
    FunctionTool(generate_interview_questions),
    FunctionTool(create_prep_plan),
]

print(f"Registered {len(tools)} tools with the agent.")

## Section 5: Create the Agent

In [ ]:
# Cell 5.1 - Define the agent
prep_coach = Agent(
    name="job_prep_coach",
    model="gemini-2.0-flash",
    description="An AI coach that helps students prepare for job and internship interviews.",
    instruction="""
    You are a supportive but honest career coach helping a student prepare for a job application.

    When given a job description and resume:
    1. Use analyze_job_description to understand what the role needs
    2. Use analyze_resume to understand what the candidate offers
    3. Identify skill gaps between the two
    4. Use generate_interview_questions to prepare the student
    5. Use create_prep_plan to build a study schedule

    Be specific, actionable, and encouraging.
    """,
    tools=tools,
)

print(f"Agent '{prep_coach.name}' created successfully!")

### How the Agent Works

Notice:
- `instruction` is the system prompt — this shapes the agent's behavior
- `tools` tells the agent what it CAN do
- **The agent decides when and how to call each tool** — we don't hardcode the order

This is the key difference from a pipeline: the agent reasons about which tool to use next!

## Section 6: Run the Agent

In [ ]:
# Cell 6.1 - Sample inputs
sample_job = """
Software Engineering Intern at Google

Requirements:
- Strong Python or Java skills required
- Experience with data structures and algorithms
- Familiarity with REST APIs
- Bonus: experience with cloud platforms (GCP preferred)
- Must communicate clearly in team settings
"""

sample_resume = """
Jane Doe | CS Junior @ State University

Skills: Python, JavaScript, HTML/CSS, some SQL

Projects:
- Built a weather app using OpenWeatherMap API
- Completed LeetCode easy/medium problems (50+)

Education: 3.6 GPA, relevant coursework in Data Structures, OOP
"""

print("Sample inputs ready for demo.")

In [ ]:
# Cell 6.2 - Run the agent
# NOTE: This cell calls the Gemini API.
# The instructor will run this live during the demo.
# If you want to try it yourself, use your own API key from aistudio.google.com

try:
    runner = InMemoryRunner(agent=prep_coach)
    session = runner.session_service.create_session(
        app_name=prep_coach.name,
        user_id="student_demo"
    )

    response = runner.run(
        user_id="student_demo",
        session_id=session.id,
        new_message=types.Content(
            role="user",
            parts=[types.Part(text=f"Job Description:\n{sample_job}\n\nMy Resume:\n{sample_resume}")]
        )
    )

    for event in response:
        if event.is_final_response():
            print(event.content.parts[0].text)
except Exception as e:
    print(f"Note: To run this cell, add your API key to cell 1.2")
    print(f"Error: {e}")

## Example Output (Pre-Run)

Here's what the agent produces (so you can follow along):

---

### Skill Gap Analysis

**Strengths:**
- ✅ Python experience (meets core requirement)
- ✅ REST API hands-on experience (weather app)
- ✅ Data structures coursework + LeetCode practice shows algorithmic thinking

**Gaps:**
- ❌ No cloud platform experience (GCP is a bonus, but valuable to learn)
- ❌ No Java experience (Python is fine, but Java shows versatility)
- ⚠️ Only 50 LeetCode problems — aim for 100+ before interviews
- ⚠️ No mention of system design concepts (important for intern interviews)

---

### Mock Interview Questions

**Technical:**
1. Walk us through your weather app project. How did you handle API rate limiting?
2. Explain the difference between a list and a linked list. When would you use each?
3. How would you design a system to cache frequently requested weather data?
4. What's the time complexity of binary search? Can you implement it?
5. Tell us about a time you optimized code for performance. What was the bottleneck?

**Behavioral:**
1. Describe a time you had to learn a new technology quickly. How did you approach it?
2. Tell us about a project where you had to collaborate with others. What was your role?
3. Give an example of constructive feedback you received and how you responded.

---

### 4-Week Prep Plan

**Week 1: Data Structures Deep Dive**
- Review arrays, strings, linked lists, hash maps
- Solve 15 LeetCode problems (easy → medium)
- Goal: solidify fundamentals

**Week 2: Algorithms & Complexity**
- Study sorting, searching, two-pointer technique
- Solve 15 LeetCode problems focusing on algorithmic patterns
- Practice explaining time/space complexity out loud

**Week 3: System Design & APIs**
- Learn REST API design best practices
- Start GCP free tier: set up a simple Cloud Function or App Engine
- Solve 10 LeetCode problems + 1 system design question

**Week 4: Mock Interviews & Polish**
- Do 2-3 mock interviews (ask friends or use Pramp.com)
- Review your weather app code — be ready to explain every line
- Practice 5 behavioral questions
- Final review of problem solutions

---

**You've got this!** You're already on the right track with Python and hands-on experience. Focus on breadth (learn Java, GCP) and depth (master data structures), and you'll be interview-ready.

## Section 7: Customize It (Your Turn!)

### Challenge Prompts

**EASY:** Change the job description or resume in cell 6.1 and re-run cell 6.2 to see how the agent adapts.

**MEDIUM:** Add a new tool to the agent. For example:
  - `suggest_portfolio_projects(skill_gaps)` — suggest projects to fill gaps
  - `rate_resume_strength(resume_text)` — score the resume and give feedback
  - `find_learning_resources(topic)` — suggest courses or tutorials

**HARD:** Modify the agent's `instruction` to change its personality:
  - Make it stricter/more critical
  - Make it more encouraging
  - Make it humorous
  
  How does the output change? What does this tell you about prompting?

In [ ]:
# Cell 7.1 - MEDIUM Challenge Starter: Add a new tool

def suggest_portfolio_projects(skill_gaps: str) -> dict:
    """
    Suggests portfolio project ideas to address skill gaps.
    """
    return {
        "skill_gaps": skill_gaps,
        "instruction": "Suggest 2-3 project ideas that would directly address these gaps and impress recruiters."
    }

# TODO: 
# 1. Uncomment the lines below
# 2. Add this tool to the tools list
# 3. Recreate the agent
# 4. Re-run cells 6.2 and 6.3

# new_tool = FunctionTool(suggest_portfolio_projects)
# tools.append(new_tool)
# print(f"Now have {len(tools)} tools!")

In [ ]:
# Cell 7.2 - HARD Challenge Starter: Change the agent's personality

# Try changing the instruction in cell 5.1 to one of these:

strict_instruction = """
You are a no-nonsense, brutally honest mentor. Be direct about weaknesses.
If the candidate is underprepared, tell them clearly. Push them to improve.
Use the tools to identify every gap and create an aggressive study plan.
"""

encouraging_instruction = """
You are an enthusiastic, warm mentor who believes in your students.
Highlight every win. Celebrate progress. Be supportive in your feedback.
Use the tools to show the candidate how much they've already accomplished.
"""

humorous_instruction = """
You are a funny, witty mentor who makes prep enjoyable.
Use humor and light callbacks to make the experience memorable.
But be thorough — hidden beneath the jokes is solid advice.
"""

# How to use: Replace the instruction in cell 5.1, run it again, then re-run cell 6.2

## Wrap-Up & Resources

### What You Just Built

You built an **AI agent** that:
- ✅ Reasons across multiple steps (not just a one-shot chatbot)
- ✅ Uses tools strategically (analyzes job, analyzes resume, generates questions, builds a plan)
- ✅ Produces personalized, actionable output based on input

---

### Where to Go Next

**Add more tools:**
- Real web search (Gemini has a built-in Google Search tool)
- Link to actual courses (Coursera, Udemy API)
- Parse real job postings from LinkedIn

**Deploy it:**
- Web UI: Gradio or Streamlit
- Backend: Cloud Run or Firebase Functions
- Production: Vertex AI Agents (Google's enterprise agent platform)

**Explore advanced agent patterns:**
- Multi-turn reasoning with memory
- Hierarchical agents (manager agent delegates to specialist agents)
- Human-in-the-loop workflows (agent suggests, human approves)

---

### Resources

- **Google ADK Docs:** [google.github.io/adk-docs](https://google.github.io/adk-docs)
- **Gemini API:** [ai.google.dev](https://ai.google.dev)
- **Free API Key:** [aistudio.google.com](https://aistudio.google.com)
- **Vertex AI Agents:** [cloud.google.com/vertex-ai/docs/agents](https://cloud.google.com/vertex-ai/docs/agents)

---

### Questions?

Post in the GDG Slack or reach out to the study jam organizers. We're excited to see what you build next!